# BBM ↔ external-platform harmonization — findings

Extends *Breakdowns in Realizing the Digital Extended Specimen*, from the paper's 131-record sample to the full BBM fungal collection, and adds automated cross-platform record resolution.

**Harmonization relationships** (README): for a BBM specimen and its counterpart on a public platform —
- **bidirectional** — we cite their id *and* they cite our catalog number back
- **unidirectional** — only one side cites the other (either direction)
- **absent** — same specimen, cited nowhere in either direction

Every number below is produced live by calling the pipeline scripts. Network-dependent cells are marked; run the fetch scripts first (`get_bbm_records.py`, `get_mo_records.py`).

In [13]:
import sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "scripts" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))
import link_audit as la
print("repo:", ROOT)

repo: /Users/wfrankel/Desktop/breakdowns_DES


## Provenance — what this run actually fetched

Read from `config` + the CSVs on disk, so the numbers below reflect **this** run,
not a remembered one. The `.env` locality/collector filters apply to **BBM only**
(`get_bbm_records.py`); the external pulls are scoped by collection / dataset /
MO seed instead. If BBM filters are ON, the "34,856 / collection-wide" figures in
§1 no longer hold — the check at the bottom of the cell flags that.

In [14]:
import csv, datetime, config as cfg

def _info(name):
    p = cfg.DATA_DIR / name
    if not p.exists():
        return "— not fetched yet"
    with open(p, encoding="utf-8") as f:
        n = sum(1 for _ in f) - 1
    ts = datetime.datetime.fromtimestamp(p.stat().st_mtime).strftime("%Y-%m-%d %H:%M")
    return f"{n:>7,} rows   (fetched {ts})"

fc, fl = cfg.FILTER_COLLECTORS or [], cfg.FILTER_LOCALITY or []
bbm_filtered = bool(fc or fl)

print("BBM filters :", "OFF — full fungal collection"
      if not bbm_filtered else f"ON — collectors={fc} locality={fl}")
print("MO seeds    :", f"user={cfg.MO_USER or '—'}  location={cfg.MO_LOCATION or '—'}")
print("data dir    :", cfg.DATA_DIR)
print()
for n in ["bbm_records.csv", "mo_records.csv", "mycoportal_records.csv",
          "gbif_records.csv", "genbank_records.csv"]:
    print(f"  {n:<24} {_info(n)}")

if bbm_filtered:
    print("\n\u26a0  BBM filters are ON \u2014 the \'34,856 / collection-wide\' "
          "figures in \u00a71 assume NO filters and no longer apply.")

BBM filters : OFF — full fungal collection
MO seeds    : user=2873  location=1679
data dir    : /Users/wfrankel/Desktop/breakdowns_DES/data

  bbm_records.csv           34,940 rows   (fetched 2026-09-03 10:56)
  mo_records.csv             5,866 rows   (fetched 2026-09-04 11:23)
  mycoportal_records.csv    34,946 rows   (fetched 2026-09-04 11:32)
  gbif_records.csv          34,878 rows   (fetched 2026-09-04 12:22)
  genbank_records.csv          500 rows   (fetched 2026-09-04 11:28)


## 1. BBM → Mushroom Observer (lookup by stored id)

MO is an **independent, upstream** platform — the Ceskas posted there directly, so MO holds no copy of our GUID. The only link is the id we recorded (`MO # 82752`). We scan `bbm_records.csv` for those, look each up on MO, and check whether MO cites us back (a free-text `UBC F#` note).

In [15]:
# NETWORK: queries Mushroom Observer
mo = la.MushroomObserver()
res = la.audit(mo)                     # scan -> lookup -> classify
c = res["counts"]
on_mo = c["bidirectional"] + c["unidirectional"]
print(f"BBM records scanned     : {res['n_rows']}")
print(f"records citing an MO id  : {res['n_with_ref']}  ({100*res['n_with_ref']/res['n_rows']:.2f}%)")
print(f"distinct MO ids          : {len(res['ref_map'])}")
print(f"  resolve on MO          : {on_mo}")
print(f"    bidirectional        : {c['bidirectional']}")
print(f"    unidirectional UBC→MO: {c['unidirectional']}")
print(f"  dangling               : {c['dangling']}")

BBM records scanned     : 34856
records citing an MO id  : 21  (0.06%)
distinct MO ids          : 20
  resolve on MO          : 20
    bidirectional        : 17
    unidirectional UBC→MO: 3
  dangling               : 0


## 1a. Identifier integrity (category 02) — sub-cases

Category 02 is *not* "identifier missing" (that is 01). Per the paper (§5.1.3) it is a cross-reference that **is present but compromised**, in one of three ways:

- **wrong** — the id resolves to a *different* specimen (the MO record cites another UBC catalog number): a mis-assigned or typo'd id.
- **wrong-field** — the id sits in a free-text column, not a structured cross-reference field, so it does not propagate downstream.
- **hanging** — the id carries no recognizable prefix, so it is unreadable as a cross-reference without insider knowledge.

We now test these **per record** instead of tagging every independent-platform link. Detecting *wrong* is done per BBM citation, so it is still caught when a good record co-cites the same MO id (which grouping by MO id would otherwise mask). `hanging` is not separately detectable here — BBM stores every MO id with an `MO #` prefix, so the scanner only ever finds prefixed ids. `wrong-field` is reported (not asserted) until `MushroomObserver.reference_fields` names the Specify field that counts as a structured cross-reference; every MO id currently lives in `co_remarks`.

In [ ]:
# Category 02 — identifier-integrity sub-cases in the MO audit (§5.1.3).
# Reuses `res` from the section-1 audit; re-run that cell first if `res` is undefined.
try:
    res
except NameError:
    res = la.audit(la.MushroomObserver())            # NETWORK

resolved = sum(1 for r in res["rows"] if r["exists"])
clean = sum(1 for r in res["rows"] if r["cites_us_back"])
c02 = res["cat02"]
print(f"resolved correspondences        : {resolved}")
print(f"  clean bidirectional           : {clean}")
print(f"  02 wrong id (UBC records)      : {c02['id_wrong']}")
print(f"  02 wrong-field (asserted)      : {c02['wrong_field']}")
for w in res["wrong"]:
    print(f"      BBM {w['bbm']} -> MO#{w['ref']}: MO cites {w['back_refs']}, not {w['our_ids']}")

cols = sorted({r["ref_source_cols"] for r in res["rows"] if r["ref_source_cols"]})
print(f"\ncolumns MO ids were scanned from : {cols}")
print("(all in a free-text notes field; set MushroomObserver.reference_fields to")
print(" assert wrong-field once the structured cross-reference field is confirmed.)")

## 2. MyCoPortal (harvested — matched by GUID)

MyCoPortal is the **opposite coupling**: it is **harvested wholesale from our Specify database** via Symbiota, so every MP record carries our GUID (`occurrenceID`) and F-number (`catalogNumber`) *by propagation*. The `Mycoportal # UBC#####` strings in our records are **legacy free-text, not queryable ids** (they even ride along verbatim into MP's `occurrenceRemarks`). The reliable link is the **GUID**.

Confirmed against the live Symbiota API (`/api/v2/occurrence?occurrenceID=<guid>`):
- UBC fungi on MyCoPortal (`collid 49`): **34,946 records** ≈ our 34,856 — essentially the whole collection.
- match: MP `occurrenceID` == BBM `guid`; MP `catalogNumber` == BBM `F#`.

**Coupling contrast, quantified:** loosely-coupled MO → **0.06 %** cross-referenced; tightly-coupled MyCoPortal → **~complete**. That is the paper's central argument in two numbers. (A GUID-discovery audit that counts MP presence + harvest gaps per record is the natural next script.)

In [16]:
# OFFLINE: how many records carry the legacy 'Mycoportal #' annotation
mp = la.MyCoPortal()
ref_map, n_rows, n_with_ref = la.scan(mp, str(la.INPUT))
print(f"records with a legacy 'Mycoportal #' note : {n_with_ref} (of {n_rows})")
print("→ not a queryable id; real MP linkage is the GUID (see above)")

records with a legacy 'Mycoportal #' note : 75 (of 34856)
→ not a queryable id; real MP linkage is the GUID (see above)


## 3. Ceska / Observatory Hill quadrants — cross-platform resolution (README §2)

To answer *how many MO records are ours but unconnected* we resolve records **by attributes**, using the evaluator subsystem **vendored** into `scripts/evaluators/` (copied from the orchestration framework) — `RuleBasedEvaluator` + `LLMEvaluator` — via `resolve.py`. Both platforms are shaped into orchestration's record contract, blocked by genus, and clustered; a cluster with a BBM and an MO record is a match. Each match is then scored into a quadrant by crossing the attribute match with the recorded cross-references.

**Seeds:** MO user 2873 (Ceska) = 5,866 obs; MO location 1679 (Observatory Hill) = 2,707 obs.

> Requires `bbm_records.csv` (joined) and `mo_records.csv`. Set `LLM_MODEL` in `.env` to enable the LLM tier.

In [17]:
import resolve as R, platforms as P
from collections import Counter
mo = P.PLATFORMS["mo"]
bbm_p, mo_p = R.DATA_DIR / "bbm_records.csv", R.DATA_DIR / "mo_records.csv"

if bbm_p.exists() and mo_p.exists():
    bbm_rows, bmeta = R.load_bbm(str(bbm_p), mo)
    plat_rows, pmeta = R.load_platform(str(mo_p))
    meta = {**bmeta, **pmeta}
    pairs, dups = R.resolve(bbm_rows, plat_rows, meta, use_llm=True)   # NETWORK if LLM_MODEL set
    q = Counter(R.quadrant(b, m, meta) for b, m, _ in pairs)
    how = Counter(h for _, _, h in pairs)
    print(f"BBM records            : {len(bbm_rows)}")
    print(f"MO Ceska/OH records    : {len(plat_rows)}")
    print(f"cross-platform matches : {len(pairs)}   (by method: {dict(how)})")
    for k in ("bidirectional","unidirectional_ubc_to_platform","unidirectional_platform_to_ubc","absent"):
        print(f"  {k:34} {q.get(k,0)}")
else:
    print("Run get_bbm_records.py and get_mo_records.py first, then re-run this cell.")

BBM records            : 34856
MO Ceska/OH records    : 5866
cross-platform matches : 21828   (by method: {'similar': 21828})
  bidirectional                      13
  unidirectional_ubc_to_platform     2
  unidirectional_platform_to_ubc     626
  absent                             21187


## 3a. Duplicate records (category 06) — attribute-level, within a platform

`resolve()` now also returns *same-platform* duplicate pairs: two records in one
attribute-matched cluster that share a platform (two MO observations, or two BBM
catalog entries, for one specimen). This is the **independent-platform side of
category 06** — the harvested side (duplicate GUIDs) is covered by
`guid_discovery.py` in §8 (`present_dup`). These are **candidates for review**,
not confirmed duplicates: attribute matching alone (name + date + locality) is
weak evidence, so a curator confirms before any merge. Written to
`reports/mo_duplicates.csv` by `resolve.py`.

In [ ]:
# Category 06 - attribute-level duplicate records (same platform, same cluster).
# Report CLUSTERS, not raw pairs: a cluster of k records yields k*(k-1)/2 pairs,
# so pairs overstate. These are candidates (weak attribute evidence) for review.
from collections import Counter, defaultdict
if 'dups' in dir():
    parent = {}
    def find(x):
        parent.setdefault(x, x)
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    def union(a, b):
        parent[find(a)] = find(b)
    for a, b, pl, how in dups:
        union(a, b)
    clusters = defaultdict(set)
    for a, b, pl, how in dups:
        clusters[find(a)] |= {a, b}
    sizes = sorted((len(s) for s in clusters.values()), reverse=True)
    by_plat = Counter(pl for _, _, pl, _ in dups)
    by_how  = Counter(how for _, _, _, how in dups)
    print(f'duplicate pairs (cat 06)   : {len(dups)}   by platform {dict(by_plat)}')
    print(f'duplicate clusters         : {len(sizes)}   (size-2: {sum(1 for s in sizes if s==2)}, >=5: {sum(1 for s in sizes if s>=5)})')
    print(f'largest clusters           : {sizes[:8]}')
    print(f'match tier                 : {dict(by_how)}   # strict = strong, similar = weak/candidate')
    print('NOTE: on the single-collector/single-locality Ceska-OH corpus, similar-tier')
    print('      clusters are dominated by distinct specimens sharing taxon+site.')
    print('      Confirm via identifier/LLM tier or DAP ground truth before trusting counts.')
else:
    print('Run cell 3 (§3) first.')

## 4. Figure 2 comparison

In [18]:
import pandas as pd
paper = {"UBC records citing MO":19, "bidirectional":8,
         "unidirectional (UBC→MO)":10, "dangling / wrong id":1, "bidirectional rate":"42%"}
ours  = {"UBC records citing MO":len(res["ref_map"]), "bidirectional":c["bidirectional"],
         "unidirectional (UBC→MO)":c["unidirectional"], "dangling / wrong id":c["dangling"],
         "bidirectional rate":f"{100*c['bidirectional']/max(len(res['ref_map']),1):.0f}%"}
pd.DataFrame({"Kholmatova 2026 (Fig 2, Phase I)":paper, "This audit (full collection)":ours})

,"Kholmatova 2026 (Fig 2, Phase I)",This audit (full collection)
UBC records citing MO,19,20
bidirectional,8,17
unidirectional (UBC→MO),10,3
dangling / wrong id,1,0
bidirectional rate,42%,85%


## 5. GBIF (harvested — matched by GUID)

GBIF is the **same coupling as MyCoPortal**: our Specify collection is **harvested wholesale** into GBIF as a published dataset (`datasetKey ca1bcd7e-7387-42f9-81ba-1470db55e3e8`), so every GBIF occurrence carries our GUID (`occurrenceID`) and F-number (`catalogNumber`) *by propagation*. Our records almost never store a GBIF id in free text — the reliable link is again the **GUID**, discovered from GBIF's side, not cited from ours.

- **Reference direction (below):** BBM free text → GBIF id. Expected ~0 — GBIF ids are not something a curator types into a specimen record.
- **Discovery direction (the real one):** `python scripts/get_records.py --platform gbif` pulls the whole dataset by `datasetKey` (~35k occurrences) and matches each back to a BBM record by `occurrenceID == guid`. Like MyCoPortal, presence should be **~complete**, so the finding is the *harvest gap* — our records with no GBIF twin — not missing cross-references.

In [19]:
# OFFLINE: GBIF ids appearing in BBM free text (real link is the GUID, discovered from GBIF)
gbif = la.PLATFORMS["gbif"]
ref_map, n_rows, n_with_ref = la.scan(gbif, str(la.INPUT))
print(f"BBM records scanned                 : {n_rows}")
print(f"records citing a GBIF id in free text: {n_with_ref}  (harvested platform → expected ~0)")
print(f"coupling                            : {gbif.coupling}  (matched by GUID / occurrenceID)")
print(f"GBIF dataset key                    : {gbif.DATASET_KEY}")
print("→ discovery direction: python scripts/get_records.py --platform gbif  (~35k occurrences)")

BBM records scanned                 : 34856
records citing a GBIF id in free text: 0  (harvested platform → expected ~0)
coupling                            : harvested  (matched by GUID / occurrenceID)
GBIF dataset key                    : ca1bcd7e-7387-42f9-81ba-1470db55e3e8
→ discovery direction: python scripts/get_records.py --platform gbif  (~35k occurrences)


## 6. GenBank (independent — matched by stored accession)

GenBank is coupled like **Mushroom Observer**: an **independent upstream** database where the link exists only if *our* record stores the sequence **accession** (`GenBank KX691234`), and it is *bidirectional* only if the GenBank record's own definition/notes cite our `UBC F#` back. Nothing is harvested, so — exactly as the paper predicts for loosely-coupled platforms — coverage is **sparse**: only sequenced specimens have an accession at all.

- **Reference direction (below):** BBM free text → GenBank accession, then look each up via NCBI eutils and check for a `UBC F#` back-reference. This is cheap: only records that actually cite an accession trigger a lookup.
- **Discovery direction:** `python scripts/get_records.py --platform genbank` searches NCBI for `UBC` / `University of British Columbia` vouchers — the way to find sequences that exist but were never linked from our side.

In [20]:
# NETWORK: only BBM records that cite a GenBank accession are looked up (expected sparse)
gb = la.PLATFORMS["genbank"]
res = la.audit(gb)
c = res["counts"]
print(f"BBM records scanned              : {res['n_rows']}")
print(f"records citing a GenBank accession: {res['n_with_ref']}")
print(f"distinct accessions               : {res['n_ids']}")
print(f"  bidirectional (cites UBC back)  : {c['bidirectional']}")
print(f"  unidirectional (UBC→GenBank)    : {c['unidirectional']}")
print(f"  dangling (accession not found)  : {c['dangling']}")
print("→ discovery direction: python scripts/get_records.py --platform genbank")

BBM records scanned              : 34856
records citing a GenBank accession: 336
distinct accessions               : 331
  bidirectional (cites UBC back)  : 2
  unidirectional (UBC→GenBank)    : 248
  dangling (accession not found)  : 81
→ discovery direction: python scripts/get_records.py --platform genbank


## 7. Cross-platform representation — summary table (Goal 1 / paper Fig 3)

One row per external platform. **Independent** platforms (MO, GenBank) are scored by citation direction — bidirectional / unidirectional each way / wrong id — the paper's Fig-3 quality breakdown. **Harvested** platforms (MP, GBIF) carry our GUID by construction, so the meaningful axis is coverage — present / harvest gap (category 03) / duplicate (category 06). `wrong id · 02` counts UBC records whose stored MO id resolves to a *different* specimen (per BBM record — the paper's unit). OFFLINE: reflects the last discovery fetch in `data/`.

In [ ]:
# Per-platform representation table. OFFLINE — reads data/*.csv.
import csv, re
from collections import Counter
import pandas as pd
from config import DATA_DIR
from platforms import norm_catalog, MushroomObserver
UUID = re.compile(r"[0-9A-Fa-f]{8}(?:-[0-9A-Fa-f]{4}){3}-[0-9A-Fa-f]{12}")
MO = MushroomObserver()

# BBM side: GUID set + per-record MO citations (scanned from co_remarks)
bbm_guid, bbm_cites_mo = set(), []
with open(DATA_DIR / "bbm_records.csv", newline="", encoding="utf-8") as f:
    for r in csv.DictReader(f):
        g = (r.get("guid") or "").strip().upper()
        if g:
            bbm_guid.add(g)
        our = {norm_catalog(x) for x in ((r.get("catalognumber") or "").strip(),
                                         (r.get("altcatalognumber") or "").strip()) if x}
        for mid in MO.extract_refs(r.get("co_remarks") or ""):
            bbm_cites_mo.append((mid, our))

def load(name):
    p = DATA_DIR / f"{name}_records.csv"
    return list(csv.DictReader(open(p, newline="", encoding="utf-8"))) if p.exists() else None

def harvested_stats(name):
    rows = load(name)
    if rows is None:
        return None
    matched = [next((m.group(0).upper() for m in UUID.finditer(r.get("ubc_ref") or "")
                     if m.group(0).upper() in bbm_guid), None) for r in rows]
    present = {g for g in matched if g}
    return {"platform": name, "coupling": "harvested", "records": len(rows),
            "present (harvested)": len(present),
            "harvest gap \u00b7 03": len(bbm_guid - present),
            "orphan": sum(1 for g in matched if g is None),
            "duplicate \u00b7 06": len([g for g in matched if g]) - len(present)}

def mo_stats():
    rows = load("mo")
    if rows is None:
        return None
    back = {}
    for r in rows:
        mid = r["id"].split(":", 1)[1] if ":" in r["id"] else r["id"]
        back[mid] = {norm_catalog(x) for x in (r.get("ubc_ref") or "").split("; ") if x}
    bi_ids, uni_u2m, wrong = set(), 0, 0
    for mid, our in bbm_cites_mo:                    # per BBM record = the paper's unit
        b = back.get(mid)
        if b is None:
            continue
        if b & our:
            bi_ids.add(mid)
        elif b:
            wrong += 1                               # MO cites a different specimen
        else:
            uni_u2m += 1                             # MO cites nobody
    cited = {mid for mid, _ in bbm_cites_mo}
    uni_m2u = sum(1 for mid, b in back.items() if b and mid not in cited)
    return {"platform": "mo", "coupling": "independent", "records": len(rows),
            "bidirectional": len(bi_ids), "uni UBC->plat": uni_u2m,
            "uni plat->UBC \u00b7 01": uni_m2u, "wrong id \u00b7 02": wrong}

def genbank_stats():
    rows = load("genbank")
    if rows is None:
        return None
    cited = sum(1 for r in rows if str(r.get("cites_ubc")).lower() == "true")
    return {"platform": "genbank", "coupling": "independent", "records": len(rows),
            "uni plat->UBC \u00b7 01": cited}

order = ["platform", "coupling", "records", "present (harvested)", "harvest gap \u00b7 03",
         "orphan", "duplicate \u00b7 06", "bidirectional", "uni UBC->plat",
         "uni plat->UBC \u00b7 01", "wrong id \u00b7 02"]
table = pd.DataFrame([x for x in [mo_stats(), harvested_stats("mycoportal"),
                                  harvested_stats("gbif"), genbank_stats()] if x])
table = table.reindex(columns=[c for c in order if c in table.columns]).fillna("\u2014")
table

## 8. Absence from repositories (category 03) — GUID reconciliation

Harvested platforms carry our GUID, so coverage is a clean GUID join (`guid_discovery.py`, offline). **Harvest gap** = a BBM specimen absent from the platform (never published downstream). **Orphan** = a platform record whose GUID isn't in BBM (investigate). Both are category 03. The undigitized-backlog sub-case of 03 is *not* measurable here — a backlog specimen has no digital trace to join on.

In [ ]:
# Category 03 — harvest-gap / orphan coverage per harvested platform. OFFLINE.
import guid_discovery as gd
import platforms as P
import pandas as pd
from config import DATA_DIR

out = []
for name, label in [("mycoportal", "MyCoPortal"), ("gbif", "GBIF")]:
    disc = DATA_DIR / f"{name}_records.csv"
    if not disc.exists():
        continue
    res = gd.audit(P.PLATFORMS[name], str(DATA_DIR / "bbm_records.csv"), str(disc))
    c = res["counts"]
    present = c["present"] + c["present_dup"]
    out.append({"platform": label, "BBM rows": res["n_bbm"], "present": present,
                "coverage %": round(100 * present / res["n_bbm"], 1),
                "harvest gap \u00b7 03": c["harvest_gap"],
                "duplicate \u00b7 06": c["present_dup"],
                "orphan \u00b7 03": res["n_orphan"]})
pd.DataFrame(out)

## 9. Validation against the 2025 DAP ground truth (paper contribution C2)

Vivian's DAP audit hand-linked Mushroom Observer records to their UBC `F#`,
GenBank accession, and the cross-reference each still needed. `data/dap_ground_truth.csv`
is the **Observatory Hill** slice (408 records; **355 with a gold MO->F# link**).
`validate_dap.py` runs `resolve.py` and scores its MO->UBC matching against that
gold standard — the automated-resolution result the paper's C2 promises.

**Rule-based baseline (scoped smoke run):** 267/355 = **75.2%** recovered (~83% of the
323 whose BBM record is in the extract), **all via the `similar` tier** (`strict` = 0,
because MO/BBM date formats differ), **33 wrong links**, 55 unmatched. The wrong+unmatched
88 are the ambiguous middle the **LLM tier** is meant to adjudicate — run this cell with
`use_llm=True` and `LLM_MODEL` set (local Ollama) to measure the LLM's contribution.

*Caveats:* DAP is a 2025 snapshot of what still needed fixing, so MO<->F# **matching** is
stable ground truth but the 'add MO #' direction labels may be partly resolved in the 2026
BBM refetch. `--per-genus` caps decoys (recall exact; wrong-link count is vs. bounded decoys).

In [ ]:
# C2 validation. On a full-memory host, bbm_path defaults to bbm_records.csv.
import logging, importlib
logging.basicConfig(level=logging.INFO, format='%(message)s', force=True)
import validate_dap as V; importlib.reload(V)
BBM = str(V.DATA_DIR / 'bbm_records.csv')
# 1) rule-based baseline:
V.validate(use_llm=False, per_genus=150, bbm_path=BBM)
# 2) LLM-only, for a clean rules-vs-LLM comparison (needs Ollama + LLM_MODEL).
#    one LLM call per genus block, so use a smaller per_genus:
# V.validate(use_llm=False, per_genus=40, bbm_path=BBM, force_llm=True)
# 3) rule-based + LLM only on the leftovers the rules missed:
# V.validate(use_llm=True,  per_genus=150, bbm_path=BBM)

## 10. GenBank linkage — unlinked genomic data (category 01)

The paper (§5.1.2) singles out GenBank: ITS sequences derived from UBC vouchers
"frequently remain unlinked to either the Mushroom Observer or UBC lineage," and it
matters most on the UBC record. `data/genbank_ground_truth.csv` — 213 accessions from
the 2025 DAP sheet, each mapped to its UBC `F#` — is the ground truth; `genbank_audit.py`
checks whether the UBC record actually carries the accession.

**Result: 212 / 213 (99.5%) are unlinked** — only one UBC record cites its own GenBank
sequence, though all 213 vouchers exist in the collection. That is the category-01 "unlinked
genomic data" finding at collection scale. The reverse direction (does the sequence's
`specimen_voucher` cite the F#?) needs a voucher-anchored fetch: `python scripts/get_records.py
--platform genbank` (now seeded from the ground truth, so it pulls the 213 collection
sequences instead of the old broad search), then re-run this cell.

In [ ]:
import importlib, logging
logging.basicConfig(level=logging.INFO, format='%(message)s', force=True)
import genbank_audit as GB; importlib.reload(GB)
GB.main()

## Methods & caveats

- **BBM data**: full `collectionobject` table + joins to determination→taxon, collector→agent, collecting-event→locality (`get_bbm_records.py`).
- **MO reference formats caught**: `MO # 82752`, `MUOB 12345`, `Mushroom Observer observation #…`, mushroomobserver.org URLs. `MO posted as …` (no number) is a link with no id and is not looked up.
- **Coupling dictates method**: harvested-downstream platforms (MyCoPortal, GBIF) match by our GUID; independent platforms (MO, GenBank) match by the id we stored + attribute resolution.
- **Resolution**: rule-based predicates use name + exact date + locality/collector *token overlap* (cross-platform strings are formatted differently); the LLM tier adjudicates the ambiguous middle when configured.
- **Coverage**: all five platforms are wired through the shared `Platform` abstraction — MO (§1), MyCoPortal (§2), GBIF (§5), GenBank (§6), plus attribute resolution (§3). Remaining work is running the harvested-side **discovery audits** (`get_records.py --platform gbif|mycoportal`) to quantify harvest gaps, and the GenBank voucher search for unlinked sequences.